In [25]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [26]:
df = pd.read_csv('train.csv')[['Age','Fare','SibSp','Parch','Survived']]

In [27]:
df.dropna(inplace=True)

In [28]:
df['family'] = df['SibSp'] + df['Parch']

In [29]:
df.head()

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [30]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [31]:
df.shape

(714, 4)

In [32]:
df.head()

,Age,Fare,Survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [33]:
X = df.drop(columns=['Survived'])
y = df['Survived']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


### Without Binarization

In [34]:
clf = DecisionTreeClassifier()
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)
accuracy_score(y_test,y_pred)

0.6293706293706294

In [35]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6443075117370892)

### Applying Binarization

In [36]:
from sklearn.preprocessing import Binarizer

In [37]:
trf = ColumnTransformer([
    ('bin',Binarizer(copy=False),['family'])
],remainder='passthrough')

In [38]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [39]:
pd.DataFrame(X_train_trf,columns=['family','Age','Fare'])

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [44]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

0.6363636363636364

In [45]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

np.float64(0.6289710485133021)

In [46]:
from sklearn.preprocessing import Binarizer

# ── Binarization → exactly 2 bins ─────────────────────────
# threshold → everything above = 1, below or equal = 0

# Age binarizer: above 18 = adult (1), below = child (0)
age_binarizer  = Binarizer(threshold=18)

# Fare binarizer: above median fare = high (1), below = low (0)
fare_median    = X_train['Fare'].median()
fare_binarizer = Binarizer(threshold=fare_median)

X_train_bin = np.column_stack([
    age_binarizer.fit_transform(X_train[['Age']]),
    fare_binarizer.fit_transform(X_train[['Fare']])
])

X_test_bin = np.column_stack([
    age_binarizer.transform(X_test[['Age']]),
    fare_binarizer.transform(X_test[['Fare']])
])

# show result
output_bin = pd.DataFrame({
    'Age'         : X_train['Age'].values,
    'Age_binary'  : X_train_bin[:, 0].astype(int),
    'Fare'        : X_train['Fare'].values,
    'Fare_binary' : X_train_bin[:, 1].astype(int)
})
print("Binarization result:")
print(output_bin.sample(8).to_string())
print(f"\nAge threshold  : 18")
print(f"Fare threshold : {fare_median:.2f} (median)")

# accuracy with binarization
clf_bin = DecisionTreeClassifier()
clf_bin.fit(X_train_bin, y_train)
bin_acc = accuracy_score(y_test, clf_bin.predict(X_test_bin))
print(f"\nAccuracy with binarization: {bin_acc:.4f}")

Binarization result:
      Age  Age_binary     Fare  Fare_binary
197  42.0           1  13.0000            0
208  35.0           1  21.0000            1
496  29.0           1  66.6000            1
439  16.0           0   7.7750            0
545  25.0           1  13.0000            0
286  48.0           1   7.8542            0
174  18.0           0   7.7750            0
59   22.0           1   7.2250            0

Age threshold  : 18
Fare threshold : 15.75 (median)

Accuracy with binarization: 0.6294
